In [3]:
pip install Box

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install bs4

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install display

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install Pillow

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install google-cloud-translate

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install tenacity

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install streamlit pymongo

Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
import time
import random
import os
import requests
import base64
from io import BytesIO
from PIL import Image
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from IPython.display import Image, display

In [14]:
# Def webscrape function
def scrape_pinterest(search_query, num_scrolls=5):
    options = Options()
    # Pinterest is very sensitive to headless mode; keeping it False is safer
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=options)
    
    try:
        # Pinterest search URL
        url = f"https://www.pinterest.com/search/pins/?q={search_query}"
        driver.get(url)
        time.sleep(5) # Let the initial grid render

        all_urls = set() # Use a set to avoid duplicates during scrolling

        for i in range(num_scrolls):
            # 1. Capture the current state of the page
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            
            # 2. Pinterest images are usually inside <img> tags within a specific div structure
            # We look for images with the 'src' that contains '736x' (Pinterest's high-res size)
            for img in soup.find_all("img"):
                src = img.get("src")
                if src and "i.pinimg.com" in src:
                    # Upgrade thumbnail URL to higher resolution if possible
                    high_res = src.replace("236x", "736x") 
                    all_urls.add(high_res)
            
            # 3. Scroll down to trigger more loading
            driver.execute_script("window.scrollBy(0, 2000);")
            print(f"Scroll {i+1}: Found {len(all_urls)} unique images so many...")
            time.sleep(random.uniform(2, 4))

        return list(all_urls)

    finally:
        driver.quit()

#Def print function
def display_robust(urls):
    for url in urls:
        display(Image(url=url, width=300))

def translate_mymemory(text):
    """
    Translates the word you give it to English
    """
    # No key required for basic use
    url = f"https://api.mymemory.translated.net/get?q={text}&langpair=de|en"
    
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()["responseData"]["translatedText"]
    return "Error"

In [15]:

def get_storage_usage(client, db_name):
    db = client[db_name]
    usage_data = []
    
    # Iterate through collection names
    for coll_name in db.list_collection_names():
        # Run the collStats command
        stats = db.command("collStats", coll_name)
        
        # Convert bytes to MB
        storage_mb = stats.get('storageSize', 0) / 1048576
        index_mb = stats.get('totalIndexSize', 0) / 1048576
        count = stats.get('count', 0)
        
        usage_data.append({
            "Collection": coll_name,
            "Documents": count,
            "Storage (MB)": round(storage_mb, 2),
            "Indexes (MB)": round(index_mb, 2)
        })
    
    return pd.DataFrame(usage_data)

# Usage in Streamlit
client = init_connection()
df_usage = get_storage_usage(client, "your_database_name")
st.table(df_usage)

total_storage = df_usage["Storage (MB)"].sum()
st.write(f"**Total Storage Used:** {total_storage:.2f} MB / 512 MB")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/athul_rohan/Documents/PetProjects/DeutschLearningAssistant/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3747, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_64251/2662133284.py", line 25, in <module>
    client = init_connection()
             ^^^^^^^^^^^^^^^
NameError: name 'init_connection' is not defined

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/athul_rohan/Documents/PetProjects/DeutschLearningAssistant/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 2222, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/athul_rohan/Documents/PetProjects/DeutschLearningAssistant/.venv/lib/python3.12/site-packages/IPython/core/ultratb.py", line 1189, in structured_traceback
    return FormattedTB.

In [ ]:


print(translate_mymemory("Der Krankenwagen"))

In [16]:
pins = scrape_pinterest(input("What do you want to scrape?\n"), num_scrolls=1)
display_robust(pins)

Scroll 1: Found 31 unique images so many...
